# 🎬 PHẦN 1: THU THẬP DỮ LIỆU PHIM TỪ IMDB (2000 - 2026)
## File: `01_Crawl_IMDb_Data.ipynb`
---
**Mục tiêu của phần này:**
1. Kết nối trực tiếp đến luồng dữ liệu mở chính thức của IMDb (`datasets.imdbws.com`).
2. Thiết lập điều kiện lọc thu thập **20.000 bộ phim** phát hành trong giai đoạn **2000 - 2026**.
3. Lọc phim điện ảnh (`movie`, `tvMovie`), loại bỏ phim người lớn (`isAdult == 0`), lọc vote tối thiểu `>= 100`.
4. Đọc giải nén luồng mạng trực tiếp (Streaming HTTP) vào DataFrame `df_raw` mà không cần tải file lớn về ổ cứng.
5. Lưu dữ liệu thô vào thư mục `data/raw/raw_movies_2000_2026.csv`.

### 1. Khai báo thư viện phục vụ cào dữ liệu
Sử dụng `urllib.request` để gửi HTTP request, `gzip` để giải nén luồng bộ nhớ, và `pandas` để quản lý bảng.

In [ ]:
import os
import time
import gzip
import urllib.request
import pandas as pd

print('✅ Đã nạp thành công các thư viện cào dữ liệu!')

### 2. Thiết lập các thông số lọc khi cào dữ liệu
Cấu hình thu thập đúng **20.000 phim** thuộc các năm từ **2000 đến 2026**.

In [ ]:
# Tham số cấu hình
TARGET_LIMIT = 20000        # Số lượng phim cần lấy
START_YEAR = 2000           # Năm bắt đầu
END_YEAR = 2026             # Năm kết thúc
MIN_VOTES = 100             # Số lượt bình chọn tối thiểu

ALLOWED_TITLE_TYPES = {'movie', 'tvMovie'}  # Phim điện ảnh / Phim truyện
EXCLUDE_ADULT = True                         # Không lấy phim người lớn

RATINGS_URL = 'https://datasets.imdbws.com/title.ratings.tsv.gz'
BASICS_URL = 'https://datasets.imdbws.com/title.basics.tsv.gz'

print(f'🎯 Mục tiêu: {TARGET_LIMIT:,} phim | Năm: {START_YEAR} - {END_YEAR} | Votes >= {MIN_VOTES:,}')

### 3. Quét luồng Ratings lấy điểm đánh giá & xếp hạng độ phổ biến

In [ ]:
print('⏳ [Bước 1/2] Đang quét luồng IMDb Ratings qua mạng HTTP...')
t0 = time.time()

req_ratings = urllib.request.Request(RATINGS_URL, headers={'User-Agent': 'Mozilla/5.0'})
ratings = {}

with urllib.request.urlopen(req_ratings, timeout=30) as response:
    with gzip.GzipFile(fileobj=response) as gz_stream:
        gz_stream.readline()  # Bỏ qua dòng tiêu đề: tconst, averageRating, numVotes
        for line in gz_stream:
            parts = line.decode('utf-8').strip().split('\t')
            if len(parts) == 3:
                tconst = parts[0]
                votes = int(parts[2])
                if votes >= MIN_VOTES:
                    avg_rating = float(parts[1])
                    ratings[tconst] = (avg_rating, votes)

print(f'-> Đã quét {len(ratings):,} phim đạt chuẩn >= {MIN_VOTES} votes trong {time.time() - t0:.2f} giây.')

# Sắp xếp theo số lượt vote giảm dần và chọn tập ứng viên đệm (70.000 phim)
sorted_ratings = sorted(ratings.items(), key=lambda x: x[1][1], reverse=True)
candidate_ids = {item[0] for item in sorted_ratings[:70000]}
print(f'-> Đã chọn ra tập ứng viên thịnh hành nhất: {len(candidate_ids):,} mã phim.')

### 4. Quét luồng Basics lọc phim theo năm (2000 - 2026) & thể loại

In [ ]:
print(f'⏳ [Bước 2/2] Đang quét luồng IMDb Basics lọc phim phát hành {START_YEAR} - {END_YEAR}...')
t1 = time.time()

req_basics = urllib.request.Request(BASICS_URL, headers={'User-Agent': 'Mozilla/5.0'})
raw_movie_list = []

with urllib.request.urlopen(req_basics, timeout=60) as response:
    with gzip.GzipFile(fileobj=response) as gz_stream:
        gz_stream.readline()  # Bỏ qua tiêu đề
        for line in gz_stream:
            parts = line.decode('utf-8').strip().split('\t')
            tconst = parts[0]
            
            # Chỉ kiểm tra nếu phim nằm trong tập ứng viên vote cao
            if tconst in candidate_ids:
                title_type = parts[1]
                is_adult = parts[4]
                year_str = parts[5]
                
                # ÁP DỤNG ĐIỀU KIỆN LỌC:
                # 1. Thuộc loại phim điện ảnh
                # 2. Không phải phim người lớn
                # 3. Năm phát hành nằm trong khoảng 2000 - 2026
                if title_type in ALLOWED_TITLE_TYPES and is_adult == '0' and year_str != r'\N':
                    year = int(year_str)
                    if START_YEAR <= year <= END_YEAR:
                        rating, votes = ratings[tconst]
                        raw_movie_list.append({
                            'id': tconst,
                            'title': parts[2],
                            'original_title': parts[3],
                            'year': year,
                            'runtime_minutes': parts[7],
                            'genres': parts[8],
                            'rating': rating,
                            'votes': votes
                        })

print(f'-> Đã trích xuất {len(raw_movie_list):,} phim hợp lệ trong {time.time() - t1:.2f} giây.')

# Sắp xếp lần cuối theo votes giảm dần và lấy đúng số lượng mục tiêu 20.000
raw_movie_list.sort(key=lambda m: m['votes'], reverse=True)
final_movies = raw_movie_list[:TARGET_LIMIT]

# Tạo DataFrame thô
df_raw = pd.DataFrame(final_movies)
print(f'\n🎉 Đã thu thập thành công {len(df_raw):,} bộ phim!')

### 5. Quan sát dữ liệu thô vừa thu thập

In [ ]:
print(f'Kích thước dữ liệu thô: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột')
display(df_raw.head(10))

### 6. Lưu dữ liệu thô ra thư mục `data/raw/`
Lưu lại để sử dụng cho **Notebook 2 (Tiền xử lý & Ép kiểu dữ liệu)**.

In [ ]:
output_dir = 'data/raw'
os.makedirs(output_dir, exist_ok=True)

csv_raw_path = os.path.join(output_dir, 'raw_movies_2000_2026.csv')
json_raw_path = os.path.join(output_dir, 'raw_movies_2000_2026.json')

df_raw.to_csv(csv_raw_path, index=False, encoding='utf-8-sig')
df_raw.to_json(json_raw_path, orient='records', force_ascii=False, indent=2)

print(f'✅ Đã lưu file thô CSV: {csv_raw_path} ({os.path.getsize(csv_raw_path)/1024/1024:.2f} MB)')
print(f'✅ Đã lưu file thô JSON: {json_raw_path} ({os.path.getsize(json_raw_path)/1024/1024:.2f} MB)')
print('\n👉 BƯỚC TIẾP THEO: Hãy mở file `02_Data_Cleaning_And_Type_Casting.ipynb` để thực hiện ép kiểu và làm sạch dữ liệu!')